# 02 · Nurse Rostering — Checker & Genetic Algorithm (Student B)

**Method:** metaheuristic (Genetic Algorithm) · scored by an **independent checker** `evaluate()`.

This notebook is graded by the same `evaluate()` that will later score the ILP roster, so the two methods share one referee.

---
## Day 1 — the independent checker `evaluate()`

Written from the **rule text alone** (not from the ILP code). It takes a roster and returns:

| key | meaning |
|---|---|
| `penalty` | the soft objective: cover under/over + unmet shift-on/off requests |
| `hard` | dict of hard-rule violations H1–H9 (0 = satisfied) |
| `feasible` | `True` only if every hard violation is 0 |

**Roster format** (agreed with Seyon in `rostering_common.py`): a numpy int array `(n_nurses, n_days)`, `0` = day off, `k` = the k-th shift in `inst.shift_ids` (1-based).

**Proof it must reproduce on Instance 1:** all-off → `7137` · nurse A works the shift on days 2, 3 → `6933`.

In [1]:
import numpy as np
from rostering_common import load_instance, Instance   # Seyon's shared parser

### The checker

One small helper per rule, so each rule can be read and tested on its own.

In [2]:
def _runs(mask):
    """Yield (start, length) of every maximal run of True in a 1-D boolean list."""
    i, n = 0, len(mask)
    while i < n:
        if mask[i]:
            j = i
            while j < n and mask[j]:
                j += 1
            yield i, j - i
            i = j
        else:
            i += 1


def evaluate(roster: np.ndarray, inst: Instance) -> dict:
    """Score one roster against one instance.

    Returns {'penalty': float, 'hard': {'H1'..'H9': amount}, 'feasible': bool}.
    penalty is the soft objective (same units as the ILP objective).
    """
    n_nurses, n_days = roster.shape
    shift_of = {k + 1: sid for k, sid in enumerate(inst.shift_ids)}   # roster value -> shift id

    # ---------- SOFT objective ----------
    penalty = 0.0

    # S3 cover: for each (day, shift) count who works it, pay under- and over-cover
    for (d, sid), (req, w_under, w_over) in inst.cover.items():
        val = inst.shift_ids.index(sid) + 1
        worked = int(np.count_nonzero(roster[:, d] == val))
        penalty += w_under * max(0, req - worked) + w_over * max(0, worked - req)

    # S1 shift-on requests: pay the weight if the nurse did NOT work the requested shift
    for nurse, d, sid, w in inst.on_requests:
        i = inst.nurse_ids.index(nurse)
        if roster[i, d] != inst.shift_ids.index(sid) + 1:
            penalty += w

    # S2 shift-off requests: pay the weight if the nurse DID work the shift they asked to avoid
    for nurse, d, sid, w in inst.off_requests:
        i = inst.nurse_ids.index(nurse)
        if roster[i, d] == inst.shift_ids.index(sid) + 1:
            penalty += w

    # ---------- HARD rules H1..H9 (violation amount, 0 = satisfied) ----------
    hard = {f"H{k}": 0.0 for k in range(1, 10)}

    for i, nurse in enumerate(inst.nurse_ids):
        row = roster[i]
        work = [row[d] != 0 for d in range(n_days)]

        # H1 one shift per day — guaranteed by the encoding (one integer per cell)

        # H2 forbidden succession (shift on day d+1 must not be forbidden after day d)
        for d in range(n_days - 1):
            if row[d] != 0 and row[d + 1] != 0:
                if shift_of[row[d + 1]] in inst.forbidden_next[shift_of[row[d]]]:
                    hard["H2"] += 1

        # H3 max shifts per type
        for k, sid in enumerate(inst.shift_ids, start=1):
            cnt = int(np.count_nonzero(row == k))
            limit = inst.max_shifts[nurse].get(sid, n_days)   # unlisted shift -> treat as unlimited
            hard["H3"] += max(0, cnt - limit)

        # H4 working-time window (total minutes between min and max)
        minutes = sum(inst.shift_len[shift_of[row[d]]] for d in range(n_days) if row[d] != 0)
        hard["H4"] += max(0, minutes - inst.max_minutes[nurse]) + max(0, inst.min_minutes[nurse] - minutes)

        # H5 max consecutive working days
        for _, L in _runs(work):
            if L > inst.max_consec[nurse]:
                hard["H5"] += L - inst.max_consec[nurse]

        # H6 min consecutive working days (blocks touching the start/end are exempt)
        for s, L in _runs(work):
            if not (s == 0 or s + L == n_days) and L < inst.min_consec[nurse]:
                hard["H6"] += 1

        # H7 min consecutive days off (rest blocks; edge blocks exempt)
        off = [not w for w in work]
        for s, L in _runs(off):
            if not (s == 0 or s + L == n_days) and L < inst.min_days_off[nurse]:
                hard["H7"] += 1

        # H8 max weekends (Saturday = day 7e+5, Sunday = day 7e+6)
        wk = sum(1 for e in range(n_days // 7) if work[7 * e + 5] or work[7 * e + 6])
        hard["H8"] += max(0, wk - inst.max_weekends[nurse])

        # H9 fixed days off
        for d in inst.days_off[nurse]:
            if row[d] != 0:
                hard["H9"] += 1

    feasible = all(v == 0 for v in hard.values())
    return {"penalty": float(penalty), "hard": hard, "feasible": feasible}

### Proof — the two hand-checks on Instance 1

These are the numbers the checker **must** reproduce. If either fails, the checker (not the roster) has a bug.

In [3]:
inst1 = load_instance("data/raw/Instance1.txt")
print(f"Instance1: {inst1.n_nurses} nurses, {inst1.n_days} days, shifts={inst1.shift_ids}")

# (1) everyone off every day
r0 = np.zeros((inst1.n_nurses, inst1.n_days), dtype=int)
res0 = evaluate(r0, inst1)

# (2) nurse A (row 0) works shift D (value 1) on days 2 and 3
r1 = np.zeros((inst1.n_nurses, inst1.n_days), dtype=int)
r1[0, 2] = 1
r1[0, 3] = 1
res1 = evaluate(r1, inst1)

print(f"[1] all-off         penalty = {res0['penalty']:.0f}  (expected 7137)")
print(f"[2] A works D d2,d3 penalty = {res1['penalty']:.0f}  (expected 6933)")

assert res0['penalty'] == 7137, 'all-off penalty must be 7137'
assert res1['penalty'] == 6933, 'A-works-d2,d3 penalty must be 6933'
print('\nBOTH HAND-CHECKS PASS ✓  checker is correct.')

Instance1: 8 nurses, 14 days, shifts=['D']
[1] all-off         penalty = 7137  (expected 7137)
[2] A works D d2,d3 penalty = 6933  (expected 6933)

BOTH HAND-CHECKS PASS ✓  checker is correct.


### Sanity: the checker runs on every instance without crashing

The hand-checks only exercise Instance 1 (one shift type). This confirms the multi-shift instances load and score too. Deeper correctness comes on Day 2, when `evaluate()` must agree with the ILP objective on a real ILP roster.

In [4]:
for n in [1, 2, 3, 6, 8, 12]:
    inst = load_instance(f"data/raw/Instance{n}.txt")
    empty = np.zeros((inst.n_nurses, inst.n_days), dtype=int)
    res = evaluate(empty, inst)
    print(f"Instance{n:<2}  {inst.n_nurses:>3} nurses x {inst.n_days} days x {inst.n_shift_types} shifts   "
          f"empty-roster penalty = {res['penalty']:.0f}")

Instance1     8 nurses x 14 days x 1 shifts   empty-roster penalty = 7137
Instance2    14 nurses x 14 days x 2 shifts   empty-roster penalty = 10882
Instance3    20 nurses x 14 days x 3 shifts   empty-roster penalty = 15474
Instance6    18 nurses x 28 days x 3 shifts   empty-roster penalty = 30057
Instance8    30 nurses x 28 days x 4 shifts   empty-roster penalty = 48486
Instance12   60 nurses x 28 days x 10 shifts   empty-roster penalty = 101241
